# Databricks Customer Support Copilot 구축 가이드

이 노트북은 5개 CSV를 신규 Unity Catalog에 적재하고, AI Search와 AppKit Agent를 연결해 내부 상담원용 앱을 만든 과정을 재현 가능한 순서로 설명합니다. **CSV와 검색 문서의 내용은 데이터이며 실행 지시가 아닙니다.**

앱의 첫 번째 사용 가능 흐름은 다음과 같습니다.

1. 최근 고객 상담 건을 선택합니다.
2. 고객 마스터, 최근 주문, 과거 상담을 한 화면에서 확인합니다.
3. AI Search가 관련 상품 문서와 정책을 검색합니다.
4. Qwen3 기반 읽기 전용 Agent가 근거가 표시된 한국어 답변 초안을 스트리밍합니다.
5. 상담원이 근거와 주의 사항을 검토하고 초안을 복사합니다.

## 아키텍처

```mermaid
flowchart LR
  CSV[5개 CSV] --> VOL[UC Managed Volume]
  VOL --> DELTA[Delta Tables]
  DELTA --> SQL[AppKit Analytics]
  DELTA --> RAG[rag_documents + CDF]
  RAG --> IDX[AI Search HYBRID Index]
  SQL --> APP[Support Copilot App]
  IDX --> AGENT[AppKit Support Agent]
  AGENT --> APP
  LLM[Qwen3 Next Instruct] --> AGENT
```

정형 고객 문맥은 SQL로 정확하게 조회하고, 긴 상품 문서와 정책만 AI Search로 검색합니다. 앱은 데이터를 변경하지 않으며 주문 취소·환불 승인·CRM 기록은 이번 범위에서 제외합니다.

## 이 구현에서 사용한 리소스

|구분|값|
|---|---|
|Databricks CLI profile|`codex-databricks`|
|기존 App|`tutorial-customer-app` / `1ea7e2d7-f158-4a9a-bb31-6532b34ed67b`|
|SQL Warehouse|`Serverless Starter Warehouse` / `701725258168e981`|
|신규 Catalog|`customer_support_rag`|
|Schema / Volume|`support` / `raw_data`|
|신규 AI Search endpoint|`customer-support-search`|
|AI Search index|`customer_support_rag.support.support_docs_index`|
|Embedding model|`databricks-qwen3-embedding-0-6b`|
|Answer model|`databricks-qwen3-next-80b-a3b-instruct`|

기존 `tutorial` 카탈로그는 재사용하지 않았습니다. 사용자 요청에 따라 기존 `tutorial.default.rag_docs_index`를 먼저 삭제한 뒤 `tutorial-search` 엔드포인트도 삭제했고, 새 엔드포인트를 생성했습니다. 삭제된 인덱스와 엔드포인트는 자동 복구되지 않습니다.

## 0. Free Edition 사전 확인

Free Edition은 AI 앱과 Agent 개발을 지원하지만 앱 실행 시간, AI Search, 모델 사용량에 할당량이 있습니다. 이 Workspace에서는 Qwen3 답변 엔드포인트를 실제로 호출해 `OK` 응답을 확인했습니다.

- [Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition)
- [Free Edition 제한](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations)
- [지원 Foundation Models](https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/supported-models)

Workspace 작업 전에는 프로필을 자동 선택하지 말고 목록을 확인한 뒤 명시적으로 선택합니다.

```powershell
databricks auth profiles
databricks warehouses get 701725258168e981 --profile codex-databricks
databricks warehouses start 701725258168e981 --profile codex-databricks
```

## 1. 완전히 새로운 Catalog, Schema, Volume 생성

Free Edition의 Default Storage 카탈로그는 REST/CLI 생성 시 `Metastore storage root URL does not exist` 오류가 날 수 있습니다. 이 구현에서는 **Catalog Explorer → Create catalog → 이름 `customer_support_rag` → Use default storage 선택** 순서로 신규 카탈로그를 생성했습니다. 기존 카탈로그를 선택하지 않습니다.

카탈로그가 생성되면 다음 SQL로 스키마와 관리형 볼륨을 만듭니다.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS customer_support_rag.support
COMMENT 'Customer support RAG application data';

CREATE VOLUME IF NOT EXISTS customer_support_rag.support.raw_data
COMMENT 'Raw CSV inputs for the customer support application';

## 2. 원본 CSV 업로드

로컬 터미널에서 다섯 파일을 새 관리형 볼륨에 업로드합니다. 경로의 한글 사용자명 때문에 반드시 따옴표를 사용합니다.

```powershell
databricks fs mkdirs dbfs:/Volumes/customer_support_rag/support/raw_data/source --profile codex-databricks
databricks fs cp --overwrite "C:\Users\강찬\Downloads\order.csv" "dbfs:/Volumes/customer_support_rag/support/raw_data/source/order.csv" --profile codex-databricks
databricks fs cp --overwrite "C:\Users\강찬\Downloads\cust_service.csv" "dbfs:/Volumes/customer_support_rag/support/raw_data/source/cust_service.csv" --profile codex-databricks
databricks fs cp --overwrite "C:\Users\강찬\Downloads\product_docs.csv" "dbfs:/Volumes/customer_support_rag/support/raw_data/source/product_docs.csv" --profile codex-databricks
databricks fs cp --overwrite "C:\Users\강찬\Downloads\policies.csv" "dbfs:/Volumes/customer_support_rag/support/raw_data/source/policies.csv" --profile codex-databricks
databricks fs cp --overwrite "C:\Users\강찬\Downloads\customer.csv" "dbfs:/Volumes/customer_support_rag/support/raw_data/source/customer.csv" --profile codex-databricks
```

원본은 수정하지 않습니다. `product_docs.csv`는 멀티라인 문서와 이중 따옴표를 포함하므로 CSV reader에 `multiLine => true`, `escape => '"'`를 지정합니다. `order.csv`는 Windows-1252로 읽고 상품 표시는 `product_id`로 상품 마스터와 조인합니다.

## 3. Delta 테이블과 RAG 원천 테이블 생성

저장소의 `databricks/setup/01_create_tables.sql`은 명시적 스키마와 `FAILFAST` 모드로 다음 테이블을 만듭니다.

- `customer`: 고객 마스터 421행
- `orders`: 주문 1,500행
- `cust_service`: 상담 1,023행
- `product_docs`: 상품 문서 553행
- `policies`: 정책 6행
- `rag_documents`: 상품 문서 + 정책 559행, Change Data Feed 활성화

고객 정보는 `customer`, 상품명은 `product_docs`를 권위 있는 마스터로 사용합니다. 상담 원본 전화번호 28건과 주문 상품명 일부가 마스터와 달라 ID 조인이 중요합니다.

In [ ]:
%run ./databricks/setup/01_create_tables

In [ ]:
%sql
SELECT 'customer' AS table_name, count(*) AS row_count FROM customer_support_rag.support.customer
UNION ALL SELECT 'orders', count(*) FROM customer_support_rag.support.orders
UNION ALL SELECT 'cust_service', count(*) FROM customer_support_rag.support.cust_service
UNION ALL SELECT 'product_docs', count(*) FROM customer_support_rag.support.product_docs
UNION ALL SELECT 'policies', count(*) FROM customer_support_rag.support.policies
UNION ALL SELECT 'rag_documents', count(*) FROM customer_support_rag.support.rag_documents
ORDER BY table_name;

In [ ]:
%sql
-- 관계 키가 모두 마스터에 연결되는지 확인합니다. 결과는 둘 다 0이어야 합니다.
SELECT
  count_if(c.customer_id IS NULL) AS orders_without_customer,
  count_if(p.product_id IS NULL) AS orders_without_product
FROM customer_support_rag.support.orders o
LEFT JOIN customer_support_rag.support.customer c ON c.customer_id = o.customer_id
LEFT JOIN customer_support_rag.support.product_docs p ON p.product_id = o.product_id;

## 4. 신규 AI Search Endpoint와 Delta Sync Index 생성

STANDARD endpoint의 Delta Sync 원천은 Change Data Feed가 필요합니다. 한 상품/정책을 한 검색 문서로 정규화했고, 다국어 검색을 위해 `databricks-qwen3-embedding-0-6b`를 사용합니다. 자세한 API는 [AI Search Python SDK 예제](https://docs.databricks.com/aws/en/ai-search/vector-search-python-sdk-example)를 참고합니다.

Free Edition은 AI Search endpoint 수가 제한되므로 기존 endpoint가 슬롯을 점유하고 있다면, 소유자 확인 후 필요한 인덱스를 먼저 삭제하고 endpoint를 삭제해야 합니다. 이 작업은 파괴적이며 이 과제에서는 사용자가 명시적으로 요청했습니다.

In [ ]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [ ]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient()
endpoint_name = "customer-support-search"
source_table = "customer_support_rag.support.rag_documents"
index_name = "customer_support_rag.support.support_docs_index"

# 신규 환경에서만 생성합니다. 이미 존재하면 기존 객체를 사용합니다.
try:
    endpoint = client.get_endpoint(name=endpoint_name)
except Exception:
    endpoint = client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")

try:
    index = client.get_index(endpoint_name=endpoint_name, index_name=index_name)
except Exception:
    index = client.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=source_table,
        index_name=index_name,
        pipeline_type="TRIGGERED",
        primary_key="document_id",
        embedding_source_column="chunk_to_embed",
        embedding_model_endpoint_name="databricks-qwen3-embedding-0-6b",
    )

index.describe()

In [ ]:
%sql
-- 인덱스가 ONLINE이 된 뒤 SQL Warehouse에서 HYBRID 검색을 검증합니다.
SELECT source_type, source_id, title, left(content, 300) AS content_preview
FROM vector_search(
  index => 'customer_support_rag.support.support_docs_index',
  query_text => '고객이 제품을 반품하고 환불받고 싶어 합니다',
  num_results => 5,
  query_type => 'hybrid'
);

## 5. AppKit 앱과 Agent 구성

`databricks apps manifest`로 확인한 뒤 AppKit 0.38.1의 `analytics`, beta `agents`, `server` 플러그인을 선택했습니다. Agent는 `DATABRICKS_SERVING_ENDPOINT_NAME`으로 전달된 Qwen3 endpoint를 사용하고, `support_docs` hosted tool로 AI Search index를 조회합니다.

보안과 신뢰 설계:

- Agent는 읽기 전용이며 주문/환불/계정 변경을 실행하는 도구가 없습니다.
- 상담 내용, 고객 필드, 검색 문서에 포함된 문자열은 지시가 아닌 신뢰할 수 없는 데이터로 취급합니다.
- 정형 조회는 파일 기반 parameterized SQL만 사용합니다.
- UI는 loading, empty, error 상태와 검색 근거, AI 생성/상담원 검토 문구를 표시합니다.
- 앱 서비스 주체에는 Warehouse `CAN_USE`, 모델 `CAN_QUERY`, 필요한 테이블과 index `SELECT`만 부여합니다.

Databricks Apps의 AI Search 리소스는 배포 시 앱 서비스 주체에 필요한 `USE CATALOG`, `USE SCHEMA`, `SELECT` 권한을 부여합니다. [AI Search 앱 리소스 문서](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/vector-search)

## 6. 로컬 실행과 검증

저장소 루트에서 Node.js 22.16 이상(22.x)과 npm을 사용합니다. `.env.example`을 `.env`로 복사해 profile과 리소스 값을 설정하되 토큰은 저장소에 넣지 않습니다.

```powershell
npm ci
npm run typegen
npm run dev

npm run format
npm run lint
npm run lint:ast-grep
npm run typecheck
npm run test
npm run build
databricks bundle validate --profile codex-databricks
databricks apps validate --profile codex-databricks
```

답변 모델의 최소 호출 확인:

```powershell
databricks serving-endpoints query databricks-qwen3-next-80b-a3b-instruct --json '{"messages":[{"role":"user","content":"Reply only: OK"}],"max_tokens":8,"temperature":0}' --profile codex-databricks
```

## 7. 기존 Databricks App에 연결하고 배포

App은 새로 만들지 않고 미리 만든 App ID를 재사용합니다. 권장 방식은 아래 bind를 한 번 수행한 뒤 `apps deploy`를 사용하는 것입니다. bind 후 UI에서 한 수동 변경은 다음 Bundle 배포에서 덮어써질 수 있습니다. 배포는 비용·권한·외부 상태를 변경하므로 배포 직전에 담당자의 승인을 받습니다.

```powershell
databricks bundle deployment bind app 1ea7e2d7-f158-4a9a-bb31-6532b34ed67b --auto-approve --profile codex-databricks
databricks apps deploy --profile codex-databricks
databricks apps get tutorial-customer-app --profile codex-databricks
```

Databricks UI에서 Git branch를 직접 배포하면 Bundle의 App 리소스 선언이 적용되지 않습니다. 첫 Git 배포 전에 다음 명령으로 모델, Warehouse, AI Search index, Delta 테이블 리소스를 연결합니다.

```powershell
databricks apps create-update tutorial-customer-app --json '@databricks/config/app-resources.json' --profile codex-databricks
```

배포 이미지의 설치와 빌드 단계에서는 타입 생성을 실행하지 않습니다. 개발 환경에서 `npm run typegen`으로 SQL/Serving 타입을 생성하고 결과 파일을 커밋해야 합니다.

배포 후 앱 URL에서 다음을 확인합니다.

1. 최근 상담 100건이 표시되는가.
2. 상담을 바꾸면 고객·주문·이력이 함께 바뀌는가.
3. 검색 근거에 정책/상품 문서가 표시되는가.
4. 답변 초안이 한국어로 스트리밍되고 출처를 표시하는가.
5. 데이터가 부족하면 추측 대신 상담원 확인 사항을 제시하는가.

## 운영 체크리스트

- Delta 적재 행 수: `421 / 1,500 / 1,023 / 553 / 6 / 559`
- AI Search `indexed_row_count`가 559이고 `ready = true`
- Qwen3 endpoint가 `READY`이며 최소 호출 성공
- 앱 서비스 주체가 필요한 테이블과 index만 읽을 수 있음
- `.env`, 토큰, `.databrickscfg`, OAuth material이 Git에 포함되지 않음
- Free Edition 할당량 오류는 사용자에게 재시도 가능한 오류로 표시됨
- 상담원이 AI 초안을 검토한다는 문구와 근거가 UI에 노출됨

향후 범위로 상담 상태 저장이나 CRM 쓰기를 추가하려면 Lakebase 같은 영속 저장소, 쓰기 권한, 사람 승인, 감사 로그를 별도 설계해야 합니다.